In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast, re, warnings
from collections import Counter
import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud
from pathlib import Path
import json
warnings.filterwarnings('ignore')

In [3]:
# pathing data
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "indotoxic2024_annotated_data-3.jsonl"

# load data
records = []
invalid_rows = 0
with DATA_PATH.open("r", encoding="utf-8") as file:
    for line in file:
        try:
            record = json.loads(line)
            if isinstance(record, dict):
                records.append(record)
            else:
                invalid_rows += 1
        except json.JSONDecodeError:
            invalid_rows += 1

In [4]:
df = pd.DataFrame(records)
display(df.head(10))

,batch_id,batch_text_id,text_id,metadata_id,annotator_id,text,initial_paragraph,topic,is_noise_or_spam_text,related_to_election_2024,toxicity,profanity_obscenity,threat_incitement_to_violence,insults,identity_attack,sexually_explicit
0,2,1,2-1,123632,20,Kemaren mas sepupuku tegang bgt dari awal. Pad...,,Disabilitas,0,0,0,0,0,0,0,0
1,2,2,2-2,265496,20,Yesaya 45:25 (TB) tetapi seluruh keturunan Isr...,,Jewish,0,0,0,0,0,0,0,0
2,2,3,2-3,86783,20,China tuding AS bantu 'provokasi' Filipina di ...,,"Terpolarisasi, Tionghoa",0,0,0,0,0,0,0,0
3,2,4,2-4,98620,20,Gua ragu Israel (“negara”) yang kita lihat sek...,,"Terpolarisasi, Jewish",0,0,1,0,0,0,1,0
4,2,5,2-5,12869,20,SENENG BGTT BELIAU BIKIN GUE GILA YA ALLAH 😭😭😭😭😭,,Disabilitas,1,0,0,0,0,0,0,0
5,2,6,2-6,43631,20,Han Hyo Joo at the 28th Busan international Fi...,,Jewish,1,0,0,0,0,0,0,0
6,2,7,2-7,28262,20,"Biaya Proyek KCJB Membengkak Luar Biasa, IDEAS...",,"Terpolarisasi, Tionghoa",0,0,1,0,0,0,1,0
7,2,8,2-8,133607,20,YENI WAHED agama KRISTEN,,"Kristen, Terpolarisasi",0,0,1,0,0,0,1,0
8,2,9,2-9,5741,20,wah gila originote bgs bget,,Disabilitas,0,0,0,0,0,0,0,0
9,2,10,2-10,66985,20,RAKYAT JEMAAH DUKUNG PRESIDEN ANIES RETWEET ME...,,"Terpolarisasi, Jewish",0,1,0,0,0,0,0,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43692 entries, 0 to 43691
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   batch_id                       43692 non-null  object
 1   batch_text_id                  43692 non-null  object
 2   text_id                        43692 non-null  object
 3   metadata_id                    43692 non-null  object
 4   annotator_id                   43692 non-null  object
 5   text                           43692 non-null  object
 6   initial_paragraph              43692 non-null  object
 7   topic                          43692 non-null  object
 8   is_noise_or_spam_text          43692 non-null  int64 
 9   related_to_election_2024       43692 non-null  int64 
 10  toxicity                       43692 non-null  int64 
 11  profanity_obscenity            43692 non-null  int64 
 12  threat_incitement_to_violence  43692 non-null  int64 
 13  i

In [6]:
df["text"].duplicated().sum()

17355

In [7]:
category_cols = [
    "profanity_obscenity",
    "threat_incitement_to_violence",
    "insults",
    "identity_attack",
    "sexually_explicit"
]

df["category_count"] = df[category_cols].sum(axis=1)

pd.crosstab(
    df["toxicity"],
    df["category_count"]
)

category_count,0,1,2,3,4,5
toxicity,,,,,,
0,35918,860,11,4,0,0
1,325,4919,1376,237,34,8


In [8]:
df["toxicity"].value_counts()

toxicity
0    36793
1     6899
Name: count, dtype: int64

In [9]:
df["topic"].value_counts()

topic
Disabilitas                                  8643
Jewish                                       7784
Tionghoa                                     6641
Terpolarisasi, Jewish                        3949
UNKNOWN                                      3912
                                             ... 
Rohingya, Disabilitas, Jewish                   1
Disabilitas, Jewish, Tionghoa                   1
Jewish, Kristen, Ahmadiyah                      1
Kristen, Jewish, Terpolarisasi, Ahmadiyah       1
LGBTQ+, Terpolarisasi, Tionghoa                 1
Name: count, Length: 87, dtype: int64

In [15]:
df["initial_paragraph"].isnull().sum()

0

In [16]:
# json to csv
df.to_csv(PROJECT_ROOT / "data" / "raw" / "indotoxic2024_annotated_data-3.csv", index=False)